## Importação de Bibliotecas

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np
from datetime import datetime
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Importando PyCaret para classificação
from pycaret.classification import *

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

In [ ]:
# Parâmetros globais
TARGET = 'FPD'
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
RANDOM_STATE = 42
# Definindo o número de folds na validação cruzada
CV = 5

## Carregamento dos Dados

In [ ]:
# Carregar dados
abt01_train = pd.read_parquet(PROCESSED_DIR / 'abt01_train.parquet')
abt01_test = pd.read_parquet(PROCESSED_DIR / 'abt01_test.parquet')

print(f'✅ Dados de Treino: {abt01_train.shape[0]:,} registros × {abt01_train.shape[1]} features')
print(f'✅ Dados de Teste: {abt01_test.shape[0]:,} registros × {abt01_test.shape[1]} features')
print(f'\nDistribuição do Target (FPD):')
print(f"  Treino: {(abt01_train['FPD'].value_counts(normalize=True) * 100).round(2)}")
print(f"  Teste: {(abt01_test['FPD'].value_counts(normalize=True) * 100).round(2)}")

## Setup do PyCaret

In [ ]:
# Necessário para evitar conflito de índices duplicados no PyCaret
abt01_train = abt01_train.reset_index(drop=True)
abt01_test  = abt01_test.reset_index(drop=True)

# Setup completo para modelagem de CLASSIFICAÇÃO no PyCaret
exp_clf101 = setup(
    data=abt01_train,                     # DataFrame: Dataset que você deseja usar para modelagem.
    target= TARGET,                       # str: Nome da coluna target (DEVE ser categórica).
    #train_size=0.7,                       # float: Proporção do dataset para treinamento.
    test_data=abt01_test,                 # DataFrame opcional para teste externo.
    index=False,                          # força PyCaret a usar RangeIndex
    ordinal_features=None,                # dict: Mapeamento de colunas ordinais e sua respectiva ordem.
    numeric_features=None,                # list: Lista de colunas tratadas como numéricas.
    categorical_features=None,            # list: Lista de colunas tratadas como categóricas.
    date_features=None,                   # list: Lista de colunas de data.
    text_features=None,                   # list: Lista de colunas de texto.
    ignore_features=None,                 # list: Lista de colunas a serem ignoradas.
    preprocess=False,                     # bool: Se True, aplica todo o pipeline de preprocessamento.
    imputation_type='simple',             # str: Tipo de imputação ('simple' ou 'iterative').
    numeric_imputation='mean',            # str: Imputação para colunas numéricas ('mean' ou 'median').
    categorical_imputation='mode',        # str: Imputação para colunas categóricas ('mode' ou 'constant').
    normalize=False,                       # bool: Se True, normaliza as colunas numéricas.
    normalize_method='zscore',            # str: Método de normalização ('zscore', 'minmax', etc.).
    transformation=False,                 # bool: Se True, transforma colunas para aproximar a normalidade.
    transformation_method='yeo-johnson',  # str: Método de transformação ('yeo-johnson' ou 'box-cox').
    pca=False,                            # bool: Se True, aplica PCA para redução de dimensionalidade.
    pca_method='linear',                  # str: Método de PCA ('linear', 'kernel', 'incremental').
    pca_components=None,                  # int ou float: Número de componentes ou variância explicada.
    bin_numeric_features=None,            # list: Lista de colunas numéricas para binning.
    remove_outliers=False,                # bool: Se True, remove outliers automaticamente.
    outliers_threshold=0.05,              # float: Proporção de outliers permitida.
    remove_multicollinearity=True,        # bool: Se True, remove features multicolineares.
    multicollinearity_threshold=0.9,      # float: Limite para correlação entre variáveis.
    polynomial_features=False,            # bool: Se True, cria features polinomiais.
    polynomial_degree=2,                  # int: Grau das features polinomiais.
    feature_selection=False,              # bool: Se True, aplica seleção automática de features.
    n_features_to_select=0.6,             # float ou int: Proporção ou número de features mantidas.
    fix_imbalance=False,                  # bool: Se True, aplica balanceamento automático de classes.
    fix_imbalance_method=None,            # objeto: Método de balanceamento (default = SMOTE).
    fold=CV,                              # int: Número de folds para validação cruzada.
    fold_shuffle=True,                    # bool: Se True, embaralha os dados nos folds.
    data_split_shuffle=True,              # bool: Se True, embaralha os dados antes do split.
    n_jobs=-1,                            # int: Número de cores utilizados (-1 usa todos).
    session_id=RANDOM_STATE,              # int: Seed para reprodutibilidade.
    verbose=False,                        # bool: Se True, imprime logs detalhados.
    profile=False                         # bool: Se True, gera um profile report do dataset.
)


## Treinando diversos algoritmos de aprendizado de máquina para classificação

In [ ]:
# propósito: lista de modelos de classificação do PyCaret
model_ids = [
    'lr',         # Logistic Regression
    #'knn',       # K Neighbors Classifier
    #'nb',        # Naive Bayes
    #'dt',         # Decision Tree Classifier
    #'svm',        # Support Vector Machine
    #'rbfsvm',    # SVM (RBF Kernel)
    #'gpc',       # Gaussian Process Classifier
    #'mlp',       # Multi Layer Perceptron
    #'ridge',     # Ridge Classifier
    #'qda',       # Quadratic Discriminant Analysis
    'rf',         # Random Forest
    #'et',        # Extra Trees Classifier
    #'ada',       # AdaBoost Classifier
    'gbc',        # Gradient Boosting Classifier
    'xgboost',    # Extreme Gradient Boosting
    'lightgbm',   # Light Gradient Boosting Machine
    #'catboost'   # CatBoost Classifier
]

## Modelos Avaliados na Fase Inicial

A seleção de modelos foi orientada por performance em dados tabulares, robustez, escalabilidade e aderência a problemas clássicos de risco de crédito (ranking de mau pagador).

### Modelos Utilizados

- **Logistic Regression (lr)**  
  Baseline do projeto. Alta interpretabilidade e excelente para *sanity check* do pipeline e das variáveis.

- **Random Forest (rf)**  
  Robusto a ruído e não linearidades. Bom equilíbrio entre performance e estabilidade.

- **Gradient Boosting (gbc)**  
  Forte em dados tabulares estruturados. Captura interações complexas entre variáveis.

- **XGBoost (xgboost)**  
  Padrão de mercado em modelos de crédito. Alta capacidade preditiva e bom controle de overfitting.

- **LightGBM (lightgbm)**  
  Extremamente eficiente e escalável. Em geral entrega as melhores métricas com menor custo computacional.

### Modelos Não Utilizados nesta Etapa

Alguns algoritmos foram deliberadamente excluídos para otimizar tempo de execução e custo computacional, sem perda relevante de performance esperada:

- **SVM / RBF-SVM**: baixa escalabilidade para datasets grandes.
- **KNN**: sensível à escala e alto custo de inferência.
- **Naive Bayes / QDA**: hipóteses estatísticas restritivas para este problema.
- **Decision Tree (isolado)**: baixo poder preditivo comparado a métodos ensemble.
- **MLP / GPC**: custo elevado e ganhos marginais em dados tabulares.
- **AdaBoost / ExtraTrees**: performance geralmente inferior a GBM/XGBoost/LightGBM neste contexto.
- **CatBoost**: não priorizado por já haver categóricas tratadas previamente.

Essa abordagem garante foco nos modelos com melhor custo-benefício para ordenação de risco de crédito.


## Comparação de Modelos

In [ ]:
# Comparar modelos de classificação e selecionar top 3 pelo AUC (foco em separar bons/mau pagadores)
print('\n🤖 Comparando modelos... (isso pode levar alguns minutos)')

compared_models = compare_models(
    include=model_ids,             # Lista de modelos de classificação
    # exclude=None,                # Modelos a excluir
    fold=CV,                       # Número de dobras da validação cruzada
    round=3,                       # Casas decimais nas métricas
    cross_validation=True,         # Executa validação cruzada
    sort='AUC',                    # Métrica para ordenação dos modelos
    n_select=3,                    # Quantidade de melhores modelos retornados
    # budget_time=None,            # Tempo máximo (em minutos) para a comparação
    turbo=True,                    # Exclui modelos mais lentos automaticamente
    errors='raise',                # Ignora falhas durante o treinamento
    # fit_kwargs=None,             # Argumentos extras passados ao método fit
    # groups=None,                 # Grupos para validação cruzada estratificada
    # experiment_custom_tags=None, # Tags customizadas do experimento
    # engine=None,                 # Backend / engine do modelo
    verbose=True,                  # Exibe progresso da execução
    # parallel=None                # Backend de paralelização
)

print('\n✅ Comparação de modelos concluída!')
print(f'\n📊 Top 3 modelos selecionados')

In [ ]:
results = pull()
print(results.shape)

In [ ]:
# Visualizar resultados da comparação
results = pull()
print('\n📊 RESULTADOS DA COMPARAÇÃO:')
display(results.head(10))

In [ ]:
# Salvar resultados
results.to_csv(ARTIFACT_DIR / 'pycaret_comparison.csv', index=False)
print(f'\n✅ Resultados salvos em: {ARTIFACT_DIR / "pycaret_comparison.csv"}')

## Tuning do Melhor Modelo

In [ ]:
# Pegar o melhor modelo (primeiro da lista)
best_model = best_models[0]

print(f'\n🎯 Melhor modelo: {best_model}')
print('\n🔧 Tunando hiperparâmetros... (isso pode levar vários minutos)')

# Tunar o melhor modelo
tuned_model = tune_model(
    best_model,
    optimize='F1',
    n_iter=20,  # Número de iterações
    verbose=False
)

print('\n✅ Tuning concluído!')

In [ ]:
# Visualizar resultados do tuning
tuning_results = pull()
print('\n📊 RESULTADOS DO TUNING:')
display(tuning_results)

# Salvar modelo tunado
save_model(tuned_model, MODELS_DIR / '04_pycaret_best_model')
print(f'\n✅ Modelo salvo em: {MODELS_DIR / "04_pycaret_best_model"}')

## Avaliação no Conjunto de Teste

In [ ]:
# Fazer predições no conjunto de teste
print('\n🔮 Fazendo predições no conjunto de teste...')

# Predições
predictions = predict_model(tuned_model, data=test_df)

# Extrair predições
y_pred = predictions['prediction_label'].values
y_pred_proba = predictions['prediction_score'].values if 'prediction_score' in predictions.columns else None

print('✅ Predições concluídas!')

## Métricas de Avaliação

In [ ]:
# Avaliar modelo
print('\n📊 AVALIAÇÃO DO MODELO:')

metrics = evaluate_model(y_test, y_pred, y_pred_proba)

# Relatório de classificação
from sklearn.metrics import classification_report
print('\n📋 RELATÓRIO DE CLASSIFICAÇÃO:')
print(classification_report(y_test, y_pred, target_names=['Bom (0)', 'Mau (1)']))

## Matriz de Confusão

In [ ]:
# Plotar matriz de confusão
cm = plot_confusion_matrix(y_test, y_pred, 
                           labels=['Bom (0)', 'Mau (1)'],
                           title='Matriz de Confusão - PyCaret Baseline')

# Salvar matriz
import pickle
with open(ARTIFACT_DIR / '04_confusion_matrix.pkl', 'wb') as f:
    pickle.dump(cm, f)

print(f'\n✅ Matriz de confusão salva')

## Curva ROC

In [ ]:
# Plotar curva ROC
if y_pred_proba is not None:
    plot_roc_curve(y_test, y_pred_proba, title='Curva ROC - PyCaret Baseline')
else:
    print('⚠️ Probabilidades não disponíveis para curva ROC')

## Feature Importance

In [ ]:
# Feature importance (se disponível)
try:
    if hasattr(tuned_model, 'feature_importances_'):
        feat_imp = plot_feature_importance(tuned_model, X_test.columns, top_n=20,
                                          title='Feature Importance - PyCaret Baseline')
        
        # Salvar feature importance
        feat_imp.to_csv(ARTIFACT_DIR / '04_feature_importance.csv', index=False)
        print(f'\n✅ Feature importance salva')
    else:
        print('\n⚠️ Feature importance não disponível para este modelo')
except Exception as e:
    print(f'\n⚠️ Erro ao plotar feature importance: {e}')

## Impacto Financeiro

In [ ]:
# Calcular impacto financeiro
print('\n💰 IMPACTO FINANCEIRO:')
impact = calculate_business_impact(y_test, y_pred, cost_per_line=50.0)

# Taxa de aprovação
aprovados = (y_pred == 0).sum()
taxa_aprovacao = aprovados / len(y_pred) * 100

print(f'\n📊 RESULTADOS DE NEGÓCIO:')
print(f'   Taxa de aprovação: {taxa_aprovacao:.2f}%')
print(f'   Clientes aprovados: {aprovados:,}')
print(f'   Clientes rejeitados: {(y_pred == 1).sum():,}')

## Salvamento de Resultados

In [ ]:
# Salvar predições
predictions_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred
})

if y_pred_proba is not None:
    predictions_df['y_pred_proba'] = y_pred_proba

predictions_df.to_csv(PREDICTIONS_DIR / '04_pycaret_baseline_predictions.csv', index=False)
print(f'\n✅ Predições salvas em: {PREDICTIONS_DIR / "04_pycaret_baseline_predictions.csv"}')

# Salvar métricas
import json

metrics_dict = {
    'model': str(tuned_model),
    'accuracy': float(metrics['accuracy']),
    'precision': float(metrics['precision']),
    'recall': float(metrics['recall']),
    'f1_score': float(metrics['f1_score']),
    'roc_auc': float(metrics.get('roc_auc', 0)),
    'taxa_aprovacao': float(taxa_aprovacao),
    'aprovados': int(aprovados),
    'ganho': float(impact['ganho']),
    'perda_inadimplencia': float(impact['perda_inadimplencia']),
    'perda_oportunidade': float(impact['perda_oportunidade']),
    'resultado_liquido': float(impact['resultado_liquido'])
}

with open(METRICS_DIR / '04_pycaret_baseline_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)

print(f'✅ Métricas salvas em: {METRICS_DIR / "04_pycaret_baseline_metrics.json"}')

## Top 3 Modelos

In [ ]:
# Salvar informações dos top 3 modelos
print('\n🏆 TOP 3 MODELOS:')

top3_info = []
for i, model in enumerate(best_models[:3], 1):
    model_name = str(model)
    print(f'   {i}. {model_name}')
    top3_info.append({'rank': i, 'model': model_name})

# Salvar top 3
import pickle
with open(ARTIFACT_DIR / '04_top3_models.pkl', 'wb') as f:
    pickle.dump(best_models[:3], f)

# Salvar info como JSON
with open(ARTIFACT_DIR / '04_top3_models_info.json', 'w') as f:
    json.dump(top3_info, f, indent=4)

print(f'\n✅ Top 3 modelos salvos em: {ARTIFACT_DIR / "04_top3_models.pkl"}')

## Resumo Final

In [ ]:
print('\n' + '='*60)
print('RESUMO DO BASELINE PYCARET')
print('='*60)
print(f'\n🤖 Melhor modelo: {tuned_model}')
print(f'\n📊 Métricas:')
print(f'   Accuracy: {metrics["accuracy"]:.4f}')
print(f'   Precision: {metrics["precision"]:.4f}')
print(f'   Recall: {metrics["recall"]:.4f}')
print(f'   F1-Score: {metrics["f1_score"]:.4f}')
if 'roc_auc' in metrics:
    print(f'   ROC-AUC: {metrics["roc_auc"]:.4f}')
print(f'\n💼 Impacto de Negócio:')
print(f'   Taxa de aprovação: {taxa_aprovacao:.2f}%')
print(f'   Clientes aprovados: {aprovados:,}')
print(f'   Resultado líquido: R$ {impact["resultado_liquido"]:,.2f}')
print(f'\n✅ Baseline estabelecido com sucesso!')
print('='*60)